In [21]:
# Gerekli kütüphanelerin kurulması
!pip install -q transformers accelerate ftfy bitsandbytes
!pip install -q git+https://github.com/huggingface/diffusers.git

# Resmi eğitim betiğinin indirilmesi (2026 güncel bağlantısı)
!wget -q https://raw.githubusercontent.com/huggingface/diffusers/main/examples/lora/train_text_to_image_lora.py

import os
import torch
from google.colab import drive

# Temel değişkenlerin ayarlanması
os.environ["MODEL_NAME"] = "runwayml/stable-diffusion-v1-5" # Ana model adı
os.environ["DATASET_NAME"] = "/content/icons_dataset"       # Veri seti yolu
os.environ["OUTPUT_DIR"] = "/content/sd-icons-model"         # Çıktı dizini

print("✅ Ortam başarıyla hazırlandı!")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ Ortam başarıyla hazırlandı!


In [22]:
import zipfile
import numpy as np
from PIL import Image

# Veri dosyasına erişmek için Google Drive'ı bağla
drive.mount('/content/drive')

# Drive içerisinde dosyayı ara (Dosya adının doğruluğundan emin olun)
zip_name = 'Icons-50.npy.zip'
def find_file(name, path):
    for root, dirs, files in os.walk(path):
        if name in files: return os.path.join(root, name)
    return None

zip_path = find_file(zip_name, '/content/drive/MyDrive')

if zip_path:
    # Arşivi aç ve klasörleri hazırla
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content/extracted_icons')

    os.makedirs(os.environ["DATASET_NAME"], exist_ok=True)

    # Verileri yükle (.npy formatından okuma)
    data = np.load('/content/extracted_icons/Icons-50.npy', allow_pickle=True).item()
    images = data['image']
    subtypes = data['subtype']
    styles = data['style']

    # Başlangıç olarak ilk 1000 görseli işle
    for i in range(min(1000, len(images))):
        # Görseli (Kanal, Yükseklik, Genişlik) formatından (Y, G, K) formatına dönüştür
        img_array = images[i].transpose(1, 2, 0)
        img = Image.fromarray(img_array.astype(np.uint8))
        # Boyutu 512x512 piksele ölçekle
        img = img.resize((512, 512), Image.LANCZOS)

        # Görseli kaydet
        img.save(f"{os.environ['DATASET_NAME']}/icon_{i}.png")

        # Yapay zeka eğitimi için metin açıklamasını (caption) oluştur ve kaydet
        caption = f"a minimalist {styles[i]} style icon of {subtypes[i]}, flat vector, white background"
        with open(f"{os.environ['DATASET_NAME']}/icon_{i}.txt", "w") as f:
            f.write(caption)

    print(f"🚀 {len(os.listdir(os.environ['DATASET_NAME']))//2} adet görsel ve açıklaması başarıyla hazırlandı.")
else:
    print("❌ Drive içerisinde veri dosyası bulunamadı.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 1000 adet görsel ve açıklaması başarıyla hazırlandı.


In [23]:
# Bellek hatalarını önlemek için Accelerate kullanarak eğitimi başlatma
!accelerate launch --mixed_precision="fp16" train_text_to_image_lora.py \
--pretrained_model_name_or_path=$MODEL_NAME \
--train_data_dir=$DATASET_NAME \
--dataloader_num_workers=8 \
--resolution=512 \
--center_crop \
--random_flip \
--train_batch_size=1 \
--gradient_accumulation_steps=4 \
--max_train_steps=500 \
--learning_rate=1e-4 \
--max_grad_norm=1 \
--lr_scheduler="cosine" \
--lr_warmup_steps=0 \
--output_dir=$OUTPUT_DIR \
--checkpointing_steps=500 \
--seed=42

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
INFO:__main__:[RANK 0] Distributed environment: DistributedType.NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local

In [24]:
import os

# Define the script path
script_path = "/content/train_text_to_image_lora.py"
script_url = "https://raw.githubusercontent.com/huggingface/diffusers/main/examples/text_to_image/train_text_to_image_lora.py"

# Check if the script exists and is not empty, otherwise download it
if os.path.exists(script_path) and os.path.getsize(script_path) > 0:
    print(f"{os.path.basename(script_path)} already exists and is not empty.")
else:
    print(f"Downloading or re-downloading {os.path.basename(script_path)}...")
    !wget $script_url -O $script_path
    if os.path.exists(script_path) and os.path.getsize(script_path) > 0:
        print(f"{os.path.basename(script_path)} downloaded successfully.")
    else:
        print(f"Failed to download {os.path.basename(script_path)}. Please check the URL and your network connection.")
        # Exit or raise an error if download fails to prevent further errors
        raise FileNotFoundError(f"{script_path} could not be downloaded or is empty.")

# Verify file presence before launch
print("Contents of /content/:")
!ls -l /content/

# Bellek (RAM/VRAM) hatalarını önlemek için Accelerate kullanarak eğitimi başlatma
!accelerate launch --mixed_precision="fp16" train_text_to_image_lora.py \
  --pretrained_model_name_or_path=$MODEL_NAME \
  --train_data_dir=$DATASET_NAME \
  --dataloader_num_workers=8 \
  --resolution=512 \
  --center_crop \
  --random_flip \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=500 \
  --learning_rate=1e-4 \
  --max_grad_norm=1 \
  --lr_scheduler="cosine" \
  --lr_warmup_steps=0 \
  --output_dir=$OUTPUT_DIR \
  --checkpointing_steps=500 \
  --seed=42

train_text_to_image_lora.py already exists and is not empty.
Contents of /content/:
total 1164
drwx------ 5 root root   4096 May 12 11:53 drive
drwxr-xr-x 2 root root   4096 May 12 11:54 extracted_icons
-rw-r--r-- 1 root root 419105 May 12 12:11 generated_icon.png
drwxr-xr-x 2 root root  69632 May 12 11:54 icons_dataset
-rw-r--r-- 1 root root 634831 May 12 12:13 red_car_comparison.png
drwxr-xr-x 1 root root   4096 May  6 13:32 sample_data
drwxr-xr-x 2 root root   4096 May 12 12:22 sd-icons-model
-rw-r--r-- 1 root root  43234 May 12 12:17 train_text_to_image_lora.py
The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to 

In [25]:
import os
from google.colab import drive

# Google Drive'ı sisteme bağlama (Mount etme)
# Bu işlemden sonra Drive dosyalarınıza '/content/drive' yolundan erişebilirsiniz.
drive.mount('/content/drive')

# Çıktı klasör yolunu Google Drive içinde olacak şekilde ayarlama
os.environ["OUTPUT_DIR"] = "/content/drive/MyDrive/sd-icons-model"

# Belirtilen yolu (klasörü) oluşturma
# 'exist_ok=True' parametresi, klasör zaten varsa hata vermesini engeller.
os.makedirs(os.environ["OUTPUT_DIR"], exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
from diffusers import StableDiffusionPipeline

# Temel modeli yükleme
pipe = StableDiffusionPipeline.from_pretrained(
    os.environ["MODEL_NAME"],
    torch_dtype=torch.float16
).to("cuda")

# Eğitilen LoRA ağırlıklarını yükleme
lora_weights = os.path.join(os.environ["OUTPUT_DIR"], "pytorch_lora_weights.safetensors")
if os.path.exists(lora_weights):
    pipe.load_lora_weights(os.environ["OUTPUT_DIR"])
    print("✅ İkon ağırlıkları başarıyla yüklendi!")

# İkon oluşturma fonksiyonu
def generate_icon(item_name):
    prompt = f"a minimalist flat vector icon of {item_name}, white background, high quality"
    image = pipe(prompt, num_inference_steps=30, guidance_scale=7.5).images[0]
    return image

# Test işlemi
test_icon = "apple" # Buradaki ismi değiştirebilirsin
result = generate_icon(test_icon)
result.save("generated_icon.png")
result.show()

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/30 [00:00<?, ?it/s]